На базе баскетбольных матчей добейтесь средней абсолютной ошибки 17 и менее очков.

### Подготовка  

In [2]:
import gdown
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [3]:
# Загрузка из google облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l10/basketball.csv', None, quiet=True)

# Библиотека для работы с базами
import pandas as pd
df = pd.read_csv('basketball.csv', encoding= 'cp1251', sep=';', header=0, index_col=0) # Загружаем базу
df.head()

,TOTAL,info,Ком. 1,Ком. 2,Минута,Общая минута,Секунда,fcount,ftime
0,"98,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,0.0,1,1.0,30,81,90.0
1,"100,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,1,1.0,45,81,105.0
2,"99,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,2,2.0,0,81,120.0
3,"98,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,2,2.0,30,81,150.0
4,"95,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,3,3.0,0,81,180.0


Извлекаем текстовые данные из колонки `info` таблицы, помещаем в переменную `data_text`. Выводим длину списка:

In [4]:
data_text = df['info'].values #

len(data_text) #

52450

Задаем максимальное кол-во слов в словаре, помещаем в переменную все символы, которые хотим вычистить из текста.

 Токенизируем текстовые данные:

In [5]:
x_num = df[['Ком. 1','Ком. 2','Минута','Секунда','ftime']].astype('int').values
y = df['fcount'].astype('int').values

scaler = StandardScaler()
x_num = scaler.fit_transform(x_num)


In [6]:
maxWordsCount = 5000
max_len = 50

tokenizer = Tokenizer(
    num_words=maxWordsCount,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
    lower=True,
    oov_token='unknown'
)

tokenizer.fit_on_texts(data_text)

seq = tokenizer.texts_to_sequences(data_text)
x_text = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')

Преобразуем данные в numpy, подготовим наборы для обучения:

In [7]:
# На вход подаются числовые данные:
input_num = layers.Input(shape=(x_num.shape[1],))

x1 = layers.Dense(64, activation='relu')(input_num)

x1 = layers.BatchNormalization()(x1)

x1 = layers.Dropout(0.2)(x1)

# На вход подаются токенизированные тексты
input_text = layers.Input(shape=(max_len,))

# Embedding преобразует индексы слов в плотные векторы признаков
x2 = layers.Embedding(maxWordsCount,64)(input_text)

# Усредняем информацию по всему тексту
# и получаем единый вектор признаков
x2 = layers.GlobalAveragePooling1D()(x2)


x2 = layers.Dense( 128,activation='relu')(x2)

x2 = layers.Dropout(0.2)(x2)


# Объединяем признаки из числовой и текстовой ветки в один вектор
x = layers.concatenate([x1, x2])

# После объединения сеть обучается на общей информации из обеих веток
x = layers.Dense(128,activation='relu')(x)

x = layers.BatchNormalization()(x)

x = layers.Dropout(0.3)(x)

x = layers.Dense(64,activation='relu')(x)

output = layers.Dense(1)(x)



# Создаем многовходовую модель:
# первый вход — числовые данные
# второй вход — текст
model = Model([input_num, input_text],output)

In [8]:
model.compile(
    optimizer=Adam(learning_rate=3e-4),
    loss='mae',
    metrics=['mae']
)

In [9]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_mae',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_mae',
    patience=8,
    restore_best_weights=True
)



In [10]:
history = model.fit(
    [x_num, x_text],
    y,
    epochs=60,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[reduce_lr, early_stop],
    verbose=1
)


Epoch 1/60
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - loss: 57.3779 - mae: 57.3779 - val_loss: 13.2686 - val_mae: 13.2686 - learning_rate: 3.0000e-04
Epoch 2/60
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 20.5150 - mae: 20.5150 - val_loss: 13.3310 - val_mae: 13.3310 - learning_rate: 3.0000e-04
Epoch 3/60
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 13.3762 - mae: 13.3762 - val_loss: 14.1103 - val_mae: 14.1103 - learning_rate: 3.0000e-04
Epoch 4/60
1297/1312 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 12.6759 - mae: 12.6759
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 12.4043 - mae: 12.4043 - val_loss: 14.6197 - val_mae: 14.6197 - learning_rate: 3.0000e-04
Epoch 5/60
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 11.8789 - mae: 11.8789 - val_loss: 14.5398 - val_mae: 14.5398 - learning_rate: 1.5000e-04
Epoch 6/60
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 11.7226 - mae: 11.7226 - va

In [12]:
y_pred = model.predict([x_num, x_text]).squeeze()
mae = np.mean(np.abs(y - y_pred))

print("MAE:", mae)


1640/1640 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
MAE: 11.007982408616291
